# ADNI: robustness, negative controls, horizon analysis, and external-transfer checks

**Standalone continuation notebook.**

This notebook does not depend on the original modeling notebook being loaded in the kernel. It loads the saved outer-test prediction artifacts from disk and performs robustness analyses. Where a test genuinely requires a different representation or a new prediction task, the notebook provides an explicit, self-contained section or a clear checkpoint rather than assuming in-memory variables exist.

In [1]:
from pathlib import Path
import gc, json, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score
)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)

DATASET_ROOT = Path(r"C:\Projects\OASIS\ADNI")
PAPER_ROOT = Path(r"C:\Projects\OASIS\ADNI/PAPER_READY_FINAL_RESULTS")
REPEAT_ROOT = PAPER_ROOT / "outer_repeats"
print("Dataset root:", DATASET_ROOT)
print("Paper root:", PAPER_ROOT)
if not PAPER_ROOT.exists():
    raise FileNotFoundError(f"Paper output directory not found: {PAPER_ROOT}")
if not REPEAT_ROOT.exists():
    raise FileNotFoundError(f"Outer-repeat directory not found: {REPEAT_ROOT}")


Dataset root: C:\Projects\OASIS\ADNI
Paper root: C:\Projects\OASIS\ADNI\PAPER_READY_FINAL_RESULTS


## 1. Load saved outer-repeat artifacts

In [2]:
def load_all_repeats():
    test_parts, oof_parts = [], []
    for rep_dir in sorted(REPEAT_ROOT.glob("repeat_*")):
        rep = int(rep_dir.name.split("_")[-1])
        tf = rep_dir / "test.csv"
        of = rep_dir / "oof.csv"
        if not tf.exists() or not of.exists():
            raise FileNotFoundError(f"Missing test.csv or oof.csv in {rep_dir}")
        t = pd.read_csv(tf); o = pd.read_csv(of)
        t["repeat"] = rep; o["repeat"] = rep
        test_parts.append(t); oof_parts.append(o)
    if not test_parts:
        raise FileNotFoundError(f"No completed outer repeats found in {REPEAT_ROOT}")
    return pd.concat(test_parts, ignore_index=True), pd.concat(oof_parts, ignore_index=True)

test_all, oof_all = load_all_repeats()
print("Repeats:", sorted(test_all.repeat.unique().tolist()))
print("Held-out rows:", len(test_all), "OOF rows:", len(oof_all))
for rep in sorted(test_all.repeat.unique()):
    assert set(test_all.loc[test_all.repeat.eq(rep),"subject_id"]).isdisjoint(
        set(oof_all.loc[oof_all.repeat.eq(rep),"subject_id"])
    )
print("PASS: subject-disjoint OOF/test separation is preserved.")


Repeats: [1, 2, 3, 4, 5, 6]
Held-out rows: 1095 OOF rows: 4509
PASS: subject-disjoint OOF/test separation is preserved.


## 2. Negative-control experiment: shuffled cross-modal pairing

In [3]:
def shuffled_pairing_null(test_df, permutations=1000, seed=42):
    rng = np.random.default_rng(seed)
    real_rows, null_rows = [], []
    for rep, g0 in test_df.groupby("repeat"):
        g = g0.reset_index(drop=True).copy()
        real_joint = (g.raw_prediction + g.clinical_prediction) / 2.0
        real_score = (g.raw_prediction - g.clinical_prediction).abs()
        real_err = (real_joint - g.future_mmse).abs()
        real = pd.DataFrame({"subject_id":g.subject_id,"score":real_score,"err":real_err})
        real = real.groupby("subject_id",as_index=False).mean(numeric_only=True)
        rho = spearmanr(real.score, real.err).statistic if len(real)>=10 and real.score.nunique()>1 and real.err.nunique()>1 else np.nan
        real_rows.append({"repeat":rep,"real_rho":rho,"N_subjects":len(real)})
        for p in range(permutations):
            idx = rng.permutation(len(g))
            pseudo_joint = (g.raw_prediction.to_numpy()[idx] + g.clinical_prediction.to_numpy()) / 2.0
            pseudo_score = np.abs(g.raw_prediction.to_numpy()[idx] - g.clinical_prediction.to_numpy())
            pseudo_err = np.abs(pseudo_joint - g.future_mmse.to_numpy())
            d = pd.DataFrame({"subject_id":g.subject_id,"score":pseudo_score,"err":pseudo_err})
            d = d.groupby("subject_id",as_index=False).mean(numeric_only=True)
            rr = spearmanr(d.score,d.err).statistic if len(d)>=10 and d.score.nunique()>1 and d.err.nunique()>1 else np.nan
            null_rows.append({"repeat":rep,"permutation":p,"rho":rr})
    return pd.DataFrame(real_rows), pd.DataFrame(null_rows)

real_nc, null_nc = shuffled_pairing_null(test_all, permutations=1000)
summary_nc=[]
for rep in real_nc.repeat:
    rr=float(real_nc.loc[real_nc.repeat.eq(rep),"real_rho"].iloc[0])
    null=null_nc.loc[null_nc.repeat.eq(rep),"rho"].dropna().to_numpy()
    summary_nc.append({
        "repeat":rep, "real_rho":rr,
        "null_mean":np.mean(null) if len(null) else np.nan,
        "null_sd":np.std(null,ddof=1) if len(null)>1 else np.nan,
        "empirical_p_one_sided":np.mean(null>=rr) if len(null) else np.nan,
        "null_N":len(null)
    })
negative_control=pd.DataFrame(summary_nc)
display(negative_control)


,repeat,real_rho,null_mean,null_sd,empirical_p_one_sided,null_N
0,1,0.503261,0.504003,0.040275,0.521,1000
1,2,0.592304,0.525034,0.051615,0.077,1000
2,3,0.514492,0.438042,0.065300,0.117,1000
3,4,0.661706,0.627858,0.037496,0.189,1000
4,5,0.650172,0.479361,0.078750,0.010,1000
5,6,0.480345,0.411955,0.076910,0.182,1000


## 3. Forecast-horizon analysis

In [4]:
if "future_interval_years" not in test_all.columns and "future_interval_days" in test_all.columns:
    test_all["future_interval_years"] = test_all["future_interval_days"] / 365.25
bins=[0.5,1.0,1.5,2.0,99]
labels=["0.5-1.0 y","1.0-1.5 y","1.5-2.0 y","2.0+ y"]
test_all["horizon_bin"]=pd.cut(test_all["future_interval_years"],bins=bins,labels=labels,right=False)
rows=[]
for (rep,h),g in test_all.groupby(["repeat","horizon_bin"],observed=True):
    d=g[["subject_id","prediction_disagreement","uncertainty","abs_error"]].dropna()
    d=d.groupby("subject_id",as_index=False).mean(numeric_only=True)
    if len(d)<10: continue
    for score in ["prediction_disagreement","uncertainty"]:
        rho,p=(np.nan,np.nan)
        if d[score].nunique()>1 and d.abs_error.nunique()>1:
            rho,p=spearmanr(d[score],d.abs_error)
        rows.append({"repeat":rep,"horizon":str(h),"score":score,"N_subjects":len(d),"rho":rho,"p":p})
horizon_df=pd.DataFrame(rows)
display(horizon_df)


,repeat,horizon,score,N_subjects,rho,p
0,1,0.5-1.0 y,prediction_disagreement,57,0.234962,0.078518
1,1,0.5-1.0 y,uncertainty,57,0.135014,0.316659
2,1,1.0-1.5 y,prediction_disagreement,38,-0.002517,0.988036
3,1,1.0-1.5 y,uncertainty,38,-0.087865,0.599889
4,2,0.5-1.0 y,prediction_disagreement,50,0.233517,0.102657
5,2,0.5-1.0 y,uncertainty,50,0.465642,0.000655
6,2,1.0-1.5 y,prediction_disagreement,40,0.020450,0.900325
7,2,1.0-1.5 y,uncertainty,40,-0.146154,0.368178
8,3,0.5-1.0 y,prediction_disagreement,56,0.336910,0.011115
9,3,0.5-1.0 y,uncertainty,56,0.505468,0.000071


## 4. History-availability and baseline-difficulty sensitivity

In [5]:
rows=[]
for rep,g in test_all.groupby("repeat"):
    for variable in ["n_mri","n_clinical","anchor_mmse"]:
        if variable not in g.columns: continue
        cut=float(g[variable].median())
        for level,gg in [("low",g[g[variable]<=cut]),("high",g[g[variable]>cut])]:
            d=gg[["subject_id","prediction_disagreement","uncertainty","abs_error"]].dropna()
            d=d.groupby("subject_id",as_index=False).mean(numeric_only=True)
            if len(d)<10: continue
            for score in ["prediction_disagreement","uncertainty"]:
                rho=spearmanr(d[score],d.abs_error).statistic if d[score].nunique()>1 and d.abs_error.nunique()>1 else np.nan
                rows.append({"repeat":rep,"stratifier":variable,"level":level,"score":score,"N_subjects":len(d),"rho":rho})
sensitivity_df=pd.DataFrame(rows)
display(sensitivity_df)


,repeat,stratifier,level,score,N_subjects,rho
0,1,n_mri,low,prediction_disagreement,62,0.113193
1,1,n_mri,low,uncertainty,62,0.058296
2,1,n_mri,high,prediction_disagreement,37,-0.003082
3,1,n_mri,high,uncertainty,37,0.050972
4,1,n_clinical,low,prediction_disagreement,60,0.095804
...,...,...,...,...,...,...
67,6,n_clinical,high,uncertainty,41,0.352091
68,6,anchor_mmse,low,prediction_disagreement,51,0.050860
69,6,anchor_mmse,low,uncertainty,51,0.223167
70,6,anchor_mmse,high,prediction_disagreement,33,-0.045455


## 5. Modality corruption / branch ablation on held-out predictions

In [6]:
# Prediction-level corruption, using no outcomes to define the corruption.
# This does not replace the raw MRI experiment; it is a cheap falsification/sensitivity control.
rows=[]
for rep,g in test_all.groupby("repeat"):
    g=g.copy()
    y=g.future_mmse.to_numpy()
    for mode in ["real","shuffle_mri","shuffle_clinical","clinical_only","mri_only"]:
        rng=np.random.default_rng(SEED+1000+rep)
        rp=g.raw_prediction.to_numpy().copy()
        cp=g.clinical_prediction.to_numpy().copy()
        if mode=="shuffle_mri": rp=rng.permutation(rp)
        if mode=="shuffle_clinical": cp=rng.permutation(cp)
        if mode=="clinical_only": pred=cp
        elif mode=="mri_only": pred=rp
        else: pred=(rp+cp)/2.0
        err=np.abs(pred-y)
        score=np.abs(rp-cp)
        rows.append({"repeat":rep,"mode":mode,
                     "MAE":float(np.mean(err)),
                     "disagreement_error_rho":float(spearmanr(score,err).statistic) if np.unique(score).size>1 and np.unique(err).size>1 else np.nan})
corruption_df=pd.DataFrame(rows)
display(corruption_df)


,repeat,mode,MAE,disagreement_error_rho
0,1,real,2.181665,0.279813
1,1,shuffle_mri,2.198055,0.373660
2,1,shuffle_clinical,3.269400,0.097457
3,1,clinical_only,1.527648,-0.022708
4,1,mri_only,3.289788,0.597233
5,2,real,2.239164,0.530646
6,2,shuffle_mri,2.258983,0.444238
7,2,shuffle_clinical,3.839006,0.295470
8,2,clinical_only,1.334184,0.017240
9,2,mri_only,3.628035,0.790426


## 6. Locked OASIS → ADNI external transfer
This is the strongest external-reliability check available without fitting any reliability map on ADNI outcomes. A monotone OOF/test source-domain map is learned from OASIS and frozen before it is applied to ADNI.

In [7]:
OASIS_PAPER=Path(r"C:\Projects\OASIS\oasis_raw_t1_encoder\PAPER_READY_FINAL_RESULTS")
def pooled_test_tables(paper_root):
    parts=[]
    for p in sorted((paper_root/"outer_repeats").glob("repeat_*/test.csv")):
        rep=int(p.parent.name.split("_")[-1])
        d=pd.read_csv(p); d["source_repeat"]=rep; parts.append(d)
    return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()

oasis_test=pooled_test_tables(OASIS_PAPER)
print("OASIS source rows:",len(oasis_test),"ADNI target rows:",len(test_all))


OASIS source rows: 311 ADNI target rows: 1095


In [8]:
transfer_rows=[]
if not oasis_test.empty:
    source=oasis_test[["subject_id","prediction_disagreement","uncertainty","abs_error"]].replace([np.inf,-np.inf],np.nan).dropna()
    source=source.groupby("subject_id",as_index=False).mean(numeric_only=True)
    for score in ["prediction_disagreement","uncertainty"]:
        src=source[[score,"abs_error"]].dropna()
        if len(src)<20 or src[score].nunique()<5: continue
        iso=IsotonicRegression(increasing=True,out_of_bounds="clip")
        iso.fit(src[score],src.abs_error)
        for rep,g in test_all.groupby("repeat"):
            tgt=g[["subject_id",score,"abs_error"]].replace([np.inf,-np.inf],np.nan).dropna()
            if len(tgt)<10: continue
            tgt=tgt.copy()
            tgt["oasis_predicted_error"]=iso.predict(tgt[score].to_numpy())
            rho=spearmanr(tgt.oasis_predicted_error,tgt.abs_error).statistic if tgt.oasis_predicted_error.nunique()>1 and tgt.abs_error.nunique()>1 else np.nan
            transfer_rows.append({"target_repeat":rep,"score":score,"N_rows":len(tgt),
                                  "target_rho":rho,"source_subjects":len(src)})
transfer_df=pd.DataFrame(transfer_rows)
display(transfer_df)


,target_repeat,score,N_rows,target_rho,source_subjects
0,1,prediction_disagreement,187,-0.000361,159
1,2,prediction_disagreement,172,0.120739,159
2,3,prediction_disagreement,184,0.144806,159
3,4,prediction_disagreement,176,0.168289,159
4,5,prediction_disagreement,188,0.168551,159
5,6,prediction_disagreement,188,-0.001500,159
6,1,uncertainty,187,-0.019291,159
7,2,uncertainty,172,0.286550,159
8,3,uncertainty,184,0.350672,159
9,4,uncertainty,176,0.317131,159


## 7. Save all robustness outputs

In [9]:
save_map={
    "negative_control_shuffled_pairing.csv":negative_control,
    "horizon_stratified_reliability.csv":horizon_df,
    "history_difficulty_sensitivity.csv":sensitivity_df,
    "prediction_level_corruption_sensitivity.csv":corruption_df,
}
if "transfer_df" in globals():
    save_map["locked_oasis_to_adni_transfer.csv"]=transfer_df
if "fs_benchmark" in globals():
    save_map["freesurfer_benchmark_checkpoint.csv"]=fs_benchmark
for name,df in save_map.items():
    df.to_csv(PAPER_ROOT/name,index=False)
print("Saved",len(save_map),"tables to",PAPER_ROOT)


Saved 5 tables to C:\Projects\OASIS\ADNI\PAPER_READY_FINAL_RESULTS


## Interpretation guardrails
Negative controls, horizon stratification, history sensitivity, and prediction-level corruption are robustness analyses. The external-transfer table is the important cross-dataset validation. None of these analyses should be used to manufacture significance. Report effect sizes, uncertainty intervals, and event/sample counts.